# Fillweight Analysis: The Power of ProcessBehavior

This notebook demonstrates the power and flexibility of ProcessBehavior for analyzing fillweight data across multiple stratification strategies.

We'll analyze the same fillweight dataset three different ways:
1. **By Lane**: Understanding machine-to-machine variation (Xbar/S charts)
2. **By Phase**: Understanding temporal process changes (Xbar/S charts)
3. **By Lane × Phase**: Stratified IMR analysis for detailed time series tracking

Each analysis takes just 2-3 lines of code, yet provides:
- Automatic SDS (Sampling Design State) detection
- Appropriate control charts based on data structure
- Signal detection with WECO rules
- Group statistics and diagnostics

**Key Features**:
- **Natural Sorting**: Charts display correctly ordered labels (Lane 1, 2, 10 - not 1, 10, 2)
- **Automatic Type Conversion**: String identifiers converted to proper types
- **Smart Chart Selection**: Xbar/S for group-level analysis, IMR for stratified time series

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, '..')

from processbehavior import ProcessDataFrame

# Load fillweight data
df = pd.read_csv('../processbehavior/datasets/data/FILLWEIGHTDATA_800.csv')

print(f"Raw Dataset: {len(df)} observations")
print(f"Columns: {list(df.columns)}")

# Create ProcessDataFrame - this will handle data cleaning
pdf = ProcessDataFrame(df)

# Check for data quality issues
n_missing = df.isna().any(axis=1).sum()
n_clean = len(df.dropna())

print(f"\nData Quality Check:")
print(f"  Rows with NaN values: {n_missing}")
print(f"  Clean rows (used in analysis): {n_clean}")
print(f"  Data loss: {n_missing} rows ({n_missing/len(df)*100:.1f}%)")

print(f"\nProcessDataFrame ready with {n_clean} observations")
print(f"\nFirst few rows of raw data:")
df.head(10)

## Data Overview

Let's understand the structure of our fillweight data:

In [ ]:
print("Data Structure:")
print(f"  Lanes: {sorted(df['lane'].unique())}")
print(f"  Phases: {sorted(df['phase'].unique())}")
print(f"  Pull range: {df['pull'].min()} to {df['pull'].max()}")
print(f"  Fillweight range: {df['fill_weight'].min():.2f} to {df['fill_weight'].max():.2f}")

# Check data types (before ProcessBehavior type conversion)
print(f"\nOriginal Data Types:")
print(f"  lane: {df['lane'].dtype}")
print(f"  phase: {df['phase'].dtype}")
print(f"  pull: {df['pull'].dtype}")

# Check data quality
print(f"\nData Quality:")
print(f"  Total rows in file: {len(df)}")
print(f"  Rows with missing values: {df.isna().any(axis=1).sum()}")
print(f"  Clean rows (will be used in analysis): {len(df.dropna())}")

---

# Analysis 1: By Lane

**Question**: Is there variation between filling machines/lanes?

**Strategy**: Group by lane, track over time (pull number)

This is the classic "machine comparison" analysis in manufacturing.

In [3]:
# Create ProcessDataFrame and analyze by lane
pdf = ProcessDataFrame(df)

analysis_by_lane = pdf.analyze(
    response_var='fill_weight',
    grouping_vars=['lane'],
    time_var='pull'
)

result_by_lane = analysis_by_lane.calculate()


PROCESS BEHAVIOR ANALYSIS

📊 Detected SDS 1: Full Factorial with Complete Replication
   All factor × time cells have n ≥ 2 observations (best case for analysis)
   Replication: Full

📈 Available charts: Xbar, S, Imr
   Selected: Xbar and S Charts (Subgroup Mean and Variation) (recommended)

📋 Data Configuration:
   Response: fill_weight
   Time: pull
   Grouping: lane

✨ Analysis Capabilities:
   • VAS residuals: R1, R2, R3, R4, R5 (R2 method: exact)
   • Main effects: Yes
   • Interactions: Yes




~/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:296: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([spec.rsg_var_name], dropna=False)[spec.response_var]
~/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:307: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grid_cells = df.groupby([spec.rsg_var_name, spec.time_var], dropna=False).size()
~/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:296: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pa

### Sampling Design State (SDS)

ProcessBehavior automatically detects the data structure:

In [4]:
sds = result_by_lane.summary['sds']
print(f"Detected SDS: {sds}")
print(f"\nSDS Interpretation:")
if sds == 1:
    print("  SDS 1: Full Replication")
    print("  - Multiple observations per (lane × time) combination")
    print("  - Both Xbar and S charts available")
    print("  - Can estimate within-subgroup variation")
elif sds == 2:
    print("  SDS 2: No Replication")
    print("  - Single observation per (lane × time) combination")
    print("  - Only Xbar chart available (no S chart)")
print(f"\nAvailable Charts: {list(result_by_lane.charts.keys())}")

Detected SDS: 1

SDS Interpretation:
  SDS 1: Full Replication
  - Multiple observations per (lane × time) combination
  - Both Xbar and S charts available
  - Can estimate within-subgroup variation

Available Charts: ['Xbar', 'Sbar']


### Group Statistics

Summary statistics for each lane:

In [ ]:
# Get group-level statistics
if 'Xbar' in result_by_lane.charts:
    # Combine Xbar and Sbar data for complete statistics
    xbar_data = result_by_lane.charts['Xbar']['data'][['rsg', 'xbar', 'lcl', 'ucl']].copy()
    sbar_data = result_by_lane.charts['Sbar']['data'][['rsg', 's']].copy() if 'Sbar' in result_by_lane.charts else None
    
    if sbar_data is not None:
        lane_stats = xbar_data.merge(sbar_data, on='rsg')
    else:
        lane_stats = xbar_data
    
    # Add observation counts from ANALYSIS dataset (after data cleaning)
    lane_counts = result_by_lane.dataset.groupby('rsg').size().reset_index(name='n')
    lane_stats = lane_stats.merge(lane_counts, on='rsg')
    
    # Sort and display
    lane_stats = lane_stats.sort_values('rsg')
    print("Lane Statistics:")
    print(lane_stats.to_string(index=False))
    
    print(f"\nTotal observations by lane (after data cleaning):")
    for _, row in lane_counts.iterrows():
        print(f"  Lane {row['rsg']}: {row['n']} observations")
    print(f"Total: {lane_counts['n'].sum()} observations")

### Signal Detection

Automatic detection of out-of-control conditions:

In [ ]:
signals = result_by_lane.get_signals()
if not signals.empty:
    print(f"Signals Detected: {len(signals)} signal(s)\n")
    for i, row in signals.iterrows():
        print(f"Signal #{i+1}:")
        print(f"  Chart: {row['chart']}")
        print(f"  Group: {row['rsg']}")
        print(f"  Beyond limits: {row['beyond_limits']}")
        if row['chart'] == 'Xbar':
            print(f"  Xbar value: {row['xbar']:.3f}, Center: {row['center']:.3f}")
            print(f"  Limits: [{row['lcl']:.3f}, {row['ucl']:.3f}]")
        elif row['chart'] == 'Sbar':
            print(f"  S value: {row['s']:.3f}, Center: {row['center']:.3f}")
            print(f"  Limits: [{row['lcl']:.3f}, {row['ucl']:.3f}]")
        print()
else:
    print("✓ No signals detected - Process appears stable by lane")

### Key Insight: Lane Analysis

This analysis answers: "Do different filling lanes produce consistently different results?"

In [ ]:
signals_by_lane = result_by_lane.get_signals()
print("\n" + "="*60)
print("LANE ANALYSIS SUMMARY")
print("="*60)
print(f"Lanes analyzed: {lane_stats['rsg'].nunique()}")
print(f"Observations per lane: {lane_stats['n'].mean():.0f} (average)")
print(f"Overall mean fillweight: {lane_stats['xbar'].mean():.3f}")
print(f"Lane-to-lane variation: {lane_stats['xbar'].std():.3f}")
print(f"Signals detected: {len(signals_by_lane)}")

---

# Analysis 2: By Phase

**Question**: Did the process change over time (between phases)?

**Strategy**: Group by phase, track over time (pull number)

This reveals whether process improvements or changes had an effect.

In [8]:
# Analyze by phase
analysis_by_phase = pdf.analyze(
    response_var='fill_weight',
    grouping_vars=['phase'],
    time_var='pull'
)

result_by_phase = analysis_by_phase.calculate()

print(f"Detected SDS: {result_by_phase.summary['sds']}")
print(f"Available Charts: {list(result_by_phase.charts.keys())}")


PROCESS BEHAVIOR ANALYSIS

📊 Detected SDS 1: Full Factorial with Complete Replication
   All factor × time cells have n ≥ 2 observations (best case for analysis)
   Replication: Full

📈 Available charts: Xbar, S, Imr
   Selected: Xbar and S Charts (Subgroup Mean and Variation) (recommended)

📋 Data Configuration:
   Response: fill_weight
   Time: pull
   Grouping: phase

✨ Analysis Capabilities:
   • VAS residuals: R1, R2, R3, R4, R5 (R2 method: exact)
   • Main effects: Yes
   • Interactions: Yes


Detected SDS: 1
Available Charts: ['Xbar', 'Sbar']


~/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:296: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([spec.rsg_var_name], dropna=False)[spec.response_var]
~/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:307: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grid_cells = df.groupby([spec.rsg_var_name, spec.time_var], dropna=False).size()
~/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:296: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pa

### Phase Statistics

In [ ]:
if 'Xbar' in result_by_phase.charts:
    # Combine Xbar and Sbar data
    xbar_data = result_by_phase.charts['Xbar']['data'][['rsg', 'xbar', 'lcl', 'ucl']].copy()
    sbar_data = result_by_phase.charts['Sbar']['data'][['rsg', 's']].copy() if 'Sbar' in result_by_phase.charts else None
    
    if sbar_data is not None:
        phase_stats = xbar_data.merge(sbar_data, on='rsg')
    else:
        phase_stats = xbar_data
    
    # Add observation counts from ANALYSIS dataset
    phase_counts = result_by_phase.dataset.groupby('rsg').size().reset_index(name='n')
    phase_stats = phase_stats.merge(phase_counts, on='rsg')
    
    phase_stats = phase_stats.sort_values('rsg')
    print("Phase Statistics:")
    print(phase_stats.to_string(index=False))
    
    print(f"\nTotal observations by phase (after data cleaning):")
    for _, row in phase_counts.iterrows():
        print(f"  Phase {row['rsg']}: {row['n']} observations")
    print(f"Total: {phase_counts['n'].sum()} observations")

### Signals by Phase

In [ ]:
signals_by_phase = result_by_phase.get_signals()
if not signals_by_phase.empty:
    print(f"Signals Detected: {len(signals_by_phase)} signal(s)\n")
    for i, row in signals_by_phase.iterrows():
        print(f"Signal #{i+1}: {row['chart']} chart in Phase {row['rsg']}")
else:
    print("✓ No signals detected - Process stable across phases")

In [ ]:
signals_by_phase = result_by_phase.get_signals()
print("\n" + "="*60)
print("PHASE ANALYSIS SUMMARY")
print("="*60)
print(f"Phases analyzed: {phase_stats['rsg'].nunique()}")
print(f"Observations per phase: {phase_stats['n'].mean():.0f} (average)")
print(f"Overall mean fillweight: {phase_stats['xbar'].mean():.3f}")
print(f"Phase-to-phase variation: {phase_stats['xbar'].std():.3f}")
print(f"Signals detected: {len(signals_by_phase)}")

---

# Analysis 3: Stratified IMR by Lane × Phase

**Question**: How does each specific lane×phase combination behave over time?

**Strategy**: IMR (Individual Moving Range) charts for each lane×phase combination

**Why IMR for Stratification?**
- IMR treats each observation as an individual, perfect for tracking time series by stratum
- More useful than stratified Xbar/S which would compare groups that may not be meaningfully different
- Reveals temporal patterns within each specific combination

In [ ]:
# IMR analysis stratified by lane × phase
analysis_imr = pdf.analyze(
    response_var='fill_weight',
    grouping_vars=['lane', 'phase'],
    time_var='pull',
    analysis_type='Imr'  # Use IMR for stratified time series
)

result_imr = analysis_imr.calculate()

print(f"Detected SDS: {result_imr.summary['sds']}")
print(f"Available Charts: {list(result_imr.charts.keys())}")

### IMR Statistics by Lane × Phase

Notice the natural sorting: 1_1, 1_2, 2_1, 2_2, not 1_1, 1_2, 10_1, 10_2!

In [ ]:
if 'Imr' in result_imr.charts:
    imr_data = result_imr.charts['Imr']['data']
    
    # Get statistics by lane×phase combination
    imr_stats = imr_data.groupby('rsg').agg({
        'x': ['count', 'mean', 'std'],
        'mr': 'mean'
    }).round(3)
    
    # Flatten column names
    imr_stats.columns = ['n', 'mean', 'std', 'mean_mr']
    imr_stats = imr_stats.reset_index()
    
    print(f"Lane × Phase IMR Statistics ({len(imr_stats)} combinations):")
    print(imr_stats.to_string(index=False))
    
    print(f"\nObservation distribution:")
    print(f"  Min n: {imr_stats['n'].min()}")
    print(f"  Max n: {imr_stats['n'].max()}")
    print(f"  Mean n: {imr_stats['n'].mean():.1f}")
    print(f"  Total: {imr_stats['n'].sum()} observations")

### Signals in Stratified IMR Analysis

In [ ]:
signals_imr = result_imr.get_signals()
if not signals_imr.empty:
    print(f"Signals Detected: {len(signals_imr)} signal(s)\n")
    
    # Show first 10 signals
    print("First 10 signals:")
    for i, row in signals_imr.head(10).iterrows():
        print(f"  {row['rsg']}: beyond_limits={row['beyond_limits']}")
    
    if len(signals_imr) > 10:
        print(f"\n... and {len(signals_imr) - 10} more signals")
else:
    print("✓ No signals detected - All lane×phase combinations stable")

In [ ]:
signals_imr = result_imr.get_signals()
print("\n" + "="*60)
print("STRATIFIED IMR ANALYSIS SUMMARY")
print("="*60)
print(f"Combinations analyzed: {imr_stats['rsg'].nunique()}")
print(f"Observations per combination: {imr_stats['n'].mean():.0f} (average)")
print(f"Overall mean fillweight: {imr_stats['mean'].mean():.3f}")
print(f"Combination-to-combination variation: {imr_stats['mean'].std():.3f}")
print(f"Average moving range: {imr_stats['mean_mr'].mean():.3f}")
print(f"Signals detected: {len(signals_imr)}")

---

# Comparison Across Analysis Strategies

Let's compare what we learned from each approach:

In [ ]:
signals_by_lane = result_by_lane.get_signals()
signals_by_phase = result_by_phase.get_signals()
signals_imr = result_imr.get_signals()

comparison = pd.DataFrame([
    {
        'Strategy': 'By Lane',
        'Chart Type': 'Xbar/S',
        'Groups': lane_stats['rsg'].nunique(),
        'Avg n': f"{lane_stats['n'].mean():.0f}",
        'SDS': result_by_lane.summary['sds'],
        'Signals': len(signals_by_lane),
        'Purpose': 'Compare machines'
    },
    {
        'Strategy': 'By Phase',
        'Chart Type': 'Xbar/S',
        'Groups': phase_stats['rsg'].nunique(),
        'Avg n': f"{phase_stats['n'].mean():.0f}",
        'SDS': result_by_phase.summary['sds'],
        'Signals': len(signals_by_phase),
        'Purpose': 'Track process changes'
    },
    {
        'Strategy': 'Lane × Phase',
        'Chart Type': 'IMR',
        'Groups': imr_stats['rsg'].nunique(),
        'Avg n': f"{imr_stats['n'].mean():.0f}",
        'SDS': result_imr.summary['sds'],
        'Signals': len(signals_imr),
        'Purpose': 'Stratified time series'
    }
])

print("\n" + "="*80)
print("ANALYSIS STRATEGY COMPARISON")
print("="*80)
print(comparison.to_string(index=False))

In [ ]:
---

# Key Takeaways

## The Power of ProcessBehavior

1. **Smart Chart Selection**:
   - **Xbar/S Charts**: For comparing groups (lanes, phases)
   - **IMR Charts**: For stratified time series analysis
   - Avoid stratifying Xbar/S - it's rarely useful to compare many small groups

2. **Flexible Stratification**: Same data, multiple analytical lenses
   - By Lane: Machine comparison (Xbar/S)
   - By Phase: Temporal analysis (Xbar/S)
   - Lane × Phase: Stratified time series (IMR)

3. **Automatic Intelligence**:
   - SDS detection determines appropriate charts
   - Signal detection runs automatically
   - Type conversion ensures correct sorting

4. **Natural Sorting** (NEW!):
   - Charts display "Lane 1, Lane 2, Lane 10" not "Lane 1, Lane 10, Lane 2"
   - Works with any identifier type (numeric, string, dates)
   - No manual sorting required

5. **Data Quality Transparency**:
   - Automatic detection and removal of missing values
   - Clear reporting of data loss
   - Counts always reflect clean, analyzed data

## When to Use Each Strategy

- **Xbar/S by Lane**: Comparing machines/processes (are they different?)
- **Xbar/S by Phase**: Tracking process changes over phases/periods
- **IMR stratified**: Time series analysis for specific combinations
- **Avoid**: Stratified Xbar/S (many small groups - not useful)

## Next Steps

Try this on your own data! Just replace:
```python
pdf = ProcessDataFrame(your_dataframe)
analysis = pdf.analyze(
    response_var='your_measurement',
    grouping_vars=['your_factors'],
    time_var='your_time_column',
    analysis_type='Imr'  # Or let it auto-detect
)
```

ProcessBehavior handles the rest!